# 03 — Analisis descriptivo

**Librerias usadas:** pandas, matplotlib, seaborn

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'solineras'))
os.chdir(ROOT)
DATA = ROOT / 'data'

f = DATA / 'clean' / 'encuestas_consolidadas.parquet'
if not f.exists():
    from consolidar import consolidar
    consolidar()

df = pd.read_parquet(f)
print(f'Dataset: {df.shape}')
if 'tipo_entidad' in df.columns:
    print(df['tipo_entidad'].value_counts())
df.head()

## 1. Descriptivo

In [ ]:
num_cols = ['potencia_pico_kwp', 'num_paneles', 'capacidad_baterias_kwh',
            'consumo_total_anual_kwh', 'pr_promedio', 'satisfaccion',
            'kwh_por_kwp', 'antiguedad_anios', 'ratio_bateria_panel']
num_cols = [c for c in num_cols if c in df.columns and df[c].notna().sum() > 0]

if num_cols:
    desc = df[num_cols].describe().T
    desc['mediana'] = df[num_cols].median()
    desc['iqr'] = df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25)
    desc['nulos_pct'] = df[num_cols].isna().mean() * 100
    display(desc.round(3))
else:
    print('Sin columnas numericas con datos')

## 2. Por tipo de entidad

In [ ]:
if num_cols and 'tipo_entidad' in df.columns:
    display(df.groupby('tipo_entidad')[num_cols].median().round(2))

## 3. Boxplots

In [ ]:
if num_cols:
    n = len(num_cols)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4*rows))
    axes = axes.flatten() if rows*cols > 1 else [axes]
    for ax, col in zip(axes, num_cols):
        if 'tipo_entidad' in df.columns and df['tipo_entidad'].nunique() > 1:
            sns.boxplot(data=df, x='tipo_entidad', y=col, ax=ax)
        else:
            sns.boxplot(y=df[col], ax=ax)
        ax.set_title(col)
    for ax in axes[len(num_cols):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 4. Frecuencias categoricas

In [ ]:
cat_cols = ['tipo_entidad', 'tipo_instalacion', 'tecnologia_paneles',
            'tipo_baterias', 'seguimiento_solar', 'conexion_red']
for c in cat_cols:
    if c in df.columns:
        print(f'\n=== {c} ===')
        print(df[c].value_counts(dropna=False).head(10))